# 06 — PD Prediction Pipeline

This notebook demonstrates how the trained PD model is used for scoring.

Workflow:

`Pre-WoE applicant data → Saved WoE bins → Final selected features → Logistic Regression → Probability of Default`


In [1]:
from pathlib import Path
import sys

import pandas as pd

CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = None

for path in [CURRENT_DIR, *CURRENT_DIR.parents]:
    if (path / "src").is_dir():
        PROJECT_ROOT = path
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Could not find project root containing 'src' from {CURRENT_DIR}"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

MODEL_DIR = PROJECT_ROOT / "models" / "pd_binning"
SPLIT_DIR = PROJECT_ROOT / "data" / "processed" / "pd_split"

print("Project root:", PROJECT_ROOT)


Project root: C:\Users\Platini AGOUANET\Mes Dossiers lourds\risk-credit-scoring


In [2]:
from src.scorings.predictor import (
    load_credit_risk_pipeline,
    transform_to_woe,
    select_model_features,
    predict_pd,
)


c:\Users\Platini AGOUANET\AppData\Local\Programs\Python\Python313\Lib\site-packages\scorecardpy\germancredit.py:5: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## 1. Load the saved credit-risk pipeline

In [3]:
pipeline = load_credit_risk_pipeline(
    MODEL_DIR / "credit_risk_pipeline.pkl"
)

print(pipeline.keys())

print("\nSelected features:")
print(pipeline["selected_features"])

print(
    "\nNumber of selected features:",
    len(pipeline["selected_features"]),
)


dict_keys(['lr_model', 'woe_bins', 'selected_features', 'coefficients', 'ks_threshold', 'ks_value', 'metrics', 'scorecard_card', 'scorecard_points0', 'scorecard_odds0_good_over_bad', 'scorecard_pdo', 'model_features', 'scorecard_factor', 'scorecard_offset'])

Selected features:
['int_rate_woe', 'loan_burden_interest_woe', 'installment_to_income_ratio_woe', 'tot_cur_bal_woe', 'total_rev_hi_lim_woe', 'loan_to_income_ratio_woe', 'credit_inquiry_rate_woe', 'annual_inc_woe', 'inq_last_6mths_woe', 'term_woe', 'purpose_woe', 'tot_coll_amt_woe', 'credit_history_months_woe', 'revol_util_woe', 'initial_list_status_woe', 'dti_woe', 'verification_status_woe', 'home_ownership_woe', 'addr_state_woe', 'open_account_ratio_woe', 'investor_funding_ratio_woe', 'total_acc_woe', 'emp_length_years_woe', 'revol_bal_woe', 'installment_woe', 'mths_since_last_record_woe']

Number of selected features: 26


c:\Users\Platini AGOUANET\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.7.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


## 2. Load the pre-WoE test data

The 49 columns are expected here. They represent the variables available before WoE transformation and before final feature selection.


In [4]:
X_test = pd.read_parquet(
    SPLIT_DIR / "X_test.parquet"
)

y_test = pd.read_parquet(
    SPLIT_DIR / "y_test.parquet"
)

print("X_test:", X_test.shape)
print("y_test :", y_test.shape)


X_test: (93257, 49)
y_test : (93257, 2)


## 3. Select one applicant

In [5]:
sample_applicant = X_test.iloc[[0]].copy()

display(sample_applicant)


,source_row_index,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,home_ownership,annual_inc,...,installment_to_income_ratio,open_account_ratio,credit_inquiry_rate,delinquency_rate,emp_title_missing,loan_burden_interest,mths_since_last_record_is_missing,mths_since_last_major_derog_is_missing,mths_since_last_delinq_is_missing,open_account_inconsistency
0,395346,1800,1800,1800.0,36,14.64,62.09,C,OWN,50000.0,...,0.014902,0.21875,0.176471,0.0,0,0.52704,0,0,1,0


## 4. Apply the saved WoE transformation

In [6]:
sample_woe = transform_to_woe(
    data=sample_applicant,
    bins=pipeline["woe_bins"],
)

woe_columns = [
    column
    for column in sample_woe.columns
    if column.endswith("_woe")
]

print(
    "WoE columns created:",
    len(woe_columns),
)

display(
    sample_woe[woe_columns]
)


[INFO] converting into woe values ...
WoE columns created: 46


,inq_last_6mths_woe,funded_amnt_inv_woe,mths_since_last_record_is_missing_woe,funded_amnt_woe,installment_woe,revol_bal_woe,mths_since_last_major_derog_is_missing_woe,initial_list_status_woe,mths_since_last_record_woe,open_account_ratio_woe,...,delinq_2yrs_woe,acc_now_delinq_woe,investor_funding_ratio_woe,investor_loan_ratio_woe,credit_inquiry_rate_woe,funding_ratio_woe,delinquency_rate_woe,revol_util_woe,loan_to_income_ratio_woe,purpose_woe
0,0.463958,0.04586,-0.041587,-0.024474,-0.070218,0.096777,-0.074521,-0.232454,-0.279717,-0.092659,...,-0.000666,-0.000007,-0.036992,-0.016558,0.485233,0.0,-0.000515,-0.049454,-0.275542,-0.141079


## 5. Keep only the final model features

In [7]:
sample_model = select_model_features(
    woe_data=sample_woe,
    selected_features=pipeline["selected_features"],
)

print(
    "Final model input shape:",
    sample_model.shape,
)

print(
    "Expected number of features:",
    len(pipeline["selected_features"]),
)

display(sample_model)


Final model input shape: (1, 26)
Expected number of features: 26


,int_rate_woe,loan_burden_interest_woe,installment_to_income_ratio_woe,tot_cur_bal_woe,total_rev_hi_lim_woe,loan_to_income_ratio_woe,credit_inquiry_rate_woe,annual_inc_woe,inq_last_6mths_woe,term_woe,...,verification_status_woe,home_ownership_woe,addr_state_woe,open_account_ratio_woe,investor_funding_ratio_woe,total_acc_woe,emp_length_years_woe,revol_bal_woe,installment_woe,mths_since_last_record_woe
0,0.10128,-0.585683,-0.299354,-0.10682,0.050282,-0.275542,0.485233,0.050706,0.463958,-0.135876,...,-0.052726,0.003605,0.167643,-0.092659,-0.036992,-0.08377,-0.10215,0.096777,-0.070218,-0.279717


## 6. Predict the Probability of Default

In [8]:
sample_pd = pipeline["lr_model"].predict_proba(
    sample_model
)[:, 1][0]

print(
    f"Probability of Default: {sample_pd:.4f}"
)

if "ks_threshold" in pipeline:
    ks_threshold = pipeline["ks_threshold"]

    statistical_class = (
        "Bad"
        if sample_pd >= ks_threshold
        else "Good"
    )

    print(
        f"KS threshold: {ks_threshold:.4f}"
    )

    print(
        "Statistical class:",
        statistical_class,
    )


Probability of Default: 0.0809
KS threshold: 0.1101
Statistical class: Good


## 7. Compare with the observed outcome

This is only an illustration. The model predicts a probability, while `good_bad` is the historical observed outcome.


In [9]:
sample_index = sample_applicant.index[0]

actual_good_bad = y_test.loc[
    sample_index,
    "good_bad",
]

actual_class = (
    "Good"
    if actual_good_bad == 1
    else "Bad"
)

print("Observed class :", actual_class)
print(f"Predicted PD   : {sample_pd:.4f}")


Observed class : Good
Predicted PD   : 0.0809


## 8. Score several applicants

In [10]:
sample_batch = X_test.iloc[:10].copy()

batch_predictions = predict_pd(
    data=sample_batch,
    pipeline=pipeline,
)

display(batch_predictions)


[INFO] converting into woe values ...


c:\Users\Platini AGOUANET\AppData\Local\Programs\Python\Python313\Lib\site-packages\scorecardpy\woebin.py:327: SyntaxWarning: invalid escape sequence '\['
  brkp = lambda x: [float(re.match('^\[(.*),.+', i).group(1)) for i in x['bin']],
c:\Users\Platini AGOUANET\AppData\Local\Programs\Python\Python313\Lib\site-packages\scorecardpy\woebin.py:372: SyntaxWarning: invalid escape sequence '\['
  .assign(brkp = lambda x: [float(re.match('^\[(.*),.+', i).group(1)) for i in x['bin']])
c:\Users\Platini AGOUANET\AppData\Local\Programs\Python\Python313\Lib\site-packages\scorecardpy\woebin.py:451: SyntaxWarning: invalid escape sequence '\['
  binning_1bst_brk['bstbrkp'] = [float(re.match("^\[(.*),.+", i).group(1)) for i in binning_1bst_brk['bstbin']]
c:\Users\Platini AGOUANET\AppData\Local\Programs\Python\Python313\Lib\site-packages\scorecardpy\woebin.py:661: SyntaxWarning: invalid escape sequence '\['
  .assign(brkp = lambda x: [float(re.match('^\[(.*),.+', i).group(1)) for i in x['bin']])
c:\Use

TypeError: object of type 'float' has no len()

## Conclusion

This notebook demonstrates the operational scoring sequence:

1. load the saved pipeline;
2. start from pre-WoE applicant data;
3. apply the WoE bins learned on the training set;
4. retain only the final selected model features;
5. apply the saved logistic regression model;
6. obtain the Probability of Default;
7. optionally compare the PD with the KS statistical threshold.

No model retraining or model-performance validation is repeated here.
